In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 - Gold: Consolidação e Validação Final
# MAGIC
# MAGIC Objetivo: ler as 3 tabelas Gold geradas e apresentar o conjunto final
# MAGIC - gold.fato_alfabetizacao_municipio (granularidade: ano, município, rede)
# MAGIC - gold.visao_uf (agregação por ano, UF, rede)
# MAGIC - gold.visao_brasil (agregação por ano, rede)
# MAGIC Padrão: somente leitura + resumo executivo (sem gravação)

# COMMAND ----------

from pyspark.sql import functions as F

# 1. LEITURA DAS TRÊS TABELAS GOLD
df_fato   = spark.table("gold.fato_alfabetizacao_municipio")
df_uf     = spark.table("gold.visao_uf")
df_brasil = spark.table("gold.visao_brasil")

print(f"[INFO] gold.fato_alfabetizacao_municipio : {df_fato.count():,} registros")
print(f"[INFO] gold.visao_uf                     : {df_uf.count():,} registros")
print(f"[INFO] gold.visao_brasil                 : {df_brasil.count():,} registros")

# COMMAND ----------

# 2. VALIDAÇÃO DE INTEGRIDADE (chaves únicas)
print("=== VALIDAÇÃO DE CHAVES ===")

# Fato: (ano, id_municipio, rede)
fato_dups = df_fato.groupBy("ano", "id_municipio", "rede").count().filter(F.col("count") > 1).count()
print(f"[FATO] Duplicados na chave (ano, id_municipio, rede): {fato_dups}")

# Visão UF: (ano, estado_sigla, rede)
uf_dups = df_uf.groupBy("ano", "estado_sigla", "rede").count().filter(F.col("count") > 1).count()
print(f"[UF]   Duplicados na chave (ano, estado_sigla, rede): {uf_dups}")

# Visão Brasil: (ano, rede)
brasil_dups = df_brasil.groupBy("ano", "rede").count().filter(F.col("count") > 1).count()
print(f"[BR]   Duplicados na chave (ano, rede): {brasil_dups}")

# COMMAND ----------

# 3. RESUMO EXECUTIVO — BRASIL
print("=== VISÃO BRASIL (resumo executivo) ===")
df_brasil.select(
    "ano", "rede_nome", "total_municipios", "municipios_distintos",
    "taxa_media", "meta_media", "pct_atingiram_meta"
).orderBy("ano", "rede_nome").show(truncate=False)

# COMMAND ----------

# 4. RESUMO EXECUTIVO — POR UF (2024, ordenado por % de atingimento)
print("=== VISÃO UF — 2024 (top 10 por % atingimento) ===")
df_uf.filter(F.col("ano") == 2024) \
     .select("estado_sigla", "rede_nome", "total_municipios",
             "taxa_media", "meta_media", "pct_atingiram_meta") \
     .orderBy(F.col("pct_atingiram_meta").desc()) \
     .show(10, truncate=False)

# COMMAND ----------

# 5. INDICADOR NACIONAL — TAXA MÉDIA E ATINGIMENTO POR ANO
print("=== INDICADOR NACIONAL POR ANO ===")
df_brasil.groupBy("ano") \
    .agg(
        F.sum("total_municipios").alias("municipios"),
        F.round(F.avg("taxa_media"), 2).alias("taxa_media_br"),
        F.sum("municipios_atingiram").alias("atingiram"),
        F.round(F.sum("municipios_atingiram") / F.sum("total_municipios") * 100, 2).alias("pct_atingiram_br"),
    ) \
    .orderBy("ano") \
    .show(truncate=False)

# COMMAND ----------

# 6. CHECAGEM FINAL — TUDO OK?
print("=== CHECKLIST FINAL ===")
ok = True

if fato_dups > 0:
    ok = False
    print("[FALHA] Fato com duplicados na chave")

if uf_dups > 0:
    ok = False
    print("[FALHA] Visão UF com duplicados na chave")

if brasil_dups > 0:
    ok = False
    print("[FALHA] Visão Brasil com duplicados na chave")

if ok:
    print("[OK] Pipeline Gold completo e consistente: fato + visão UF + visão Brasil")
    print("     - 2023: sem meta (esperado) | 2024: metas preenchidas e % de atingimento calculado")